In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
from pathlib import Path

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from scraper import fetch_website_contents
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key found")
else:
    print("API KEY NOT FOUND")
    
openai = OpenAI()

!ollama pull llama3.2
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

API key found


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
links = fetch_website_links("https://www.youtube.com/")
links

['/',
 '/',
 'https://www.youtube.com/about/',
 'https://www.youtube.com/about/press/',
 'https://www.youtube.com/about/copyright/',
 '/t/contact_us/',
 'https://www.youtube.com/creators/',
 'https://www.youtube.com/ads/',
 'https://developers.google.com/youtube',
 '/t/terms',
 '/t/privacy',
 'https://www.youtube.com/about/policies/',
 'https://www.youtube.com/howyoutubeworks?utm_campaign=ytgen&utm_source=ythp&utm_medium=LeftNav&utm_content=txt&u=https%3A%2F%2Fwww.youtube.com%2Fhowyoutubeworks%3Futm_source%3Dythp%26utm_medium%3DLeftNav%26utm_campaign%3Dytgen',
 '/new']

### Usando Ollama para ler os links de uma página web e responder em JSON.  
O objetivo é identificar links relevantes em uma página, usando "one shot prompting", onde será providenciado um exemplo de como esperamos a resposta.


#### Funções e constantes para gerar o json com os links relevantes

In [ ]:
link_system_prompt = """
Segue a lista de links de um website.
Por favor, decida quais links são relevantes para se ter em um folheto/resumo sobre a compania, 
Como por exemplo seções que contam sobre a empresa, beneficios, páginas de carreira, missão, valores
Formato de resposta esperado em Json, conforme exemplo abaixo:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Segue a lista de links do website {url} -
Por favor, decida quais links são relevantes para se ter em um folheto/resumo sobre a compania, 
Responda com a URL https completa em JSON.
Por favor, não inclua termos de serviço, privacidade ou links de emails.

Links :

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


def select_relevant_links(url):
    
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Foram encontrados {len(links['links'])} links relevantes")
    return links

##### Testando as funções acima

In [10]:
select_relevant_links("https://www.youtube.com/")

{'links': [{'type': 'sobre a empresa',
   'url': 'https://www.youtube.com/about/'},
  {'type': 'press', 'url': 'https://www.youtube.com/about/press/'},
  {'type': 'creadores', 'url': 'https://www.youtube.com/creators/'},
  {'type': 'pós-produção', 'url': 'https://new.somewhere/'}]}

#### Funções e constantes para gerar o resumo

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Você está analisando uma empresa chamada: {company_name}
Segue o conteudo da sua página principal e outras páginas relevantes;
Use essas informações para construir um resumo/folheto em markdown sem blocos de código.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

brochure_system_prompt = """
Você é um assistente que tem o papel de analisar páginas relevantes de um website de uma empresa e 
criar um breve resumo sobre a compania para clientes, investidores e recrutamento.
Por favor, inclua detalhes sobre a empresa, como cultura, clientes, carreiras se tiver informação suficiente.
Responda em markdown sem blocos de código.

"""

def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

##### Testando

In [ ]:
create_brochure("Youtube", "https://www.youtube.com/")